# LLM Criticism and Human Correction

This notebook audits the Gemini critic stage and the resulting human corrections. Execution, validation, append-only batch synchronization, and workbook creation are handled by `run_gemini_criticism.py` following `gemini_criticism_runbook.md`. Gemini suggestions remain advisory: only rows explicitly reviewed by the human coder can replace Codex labels.


In [1]:
from __future__ import annotations

from math import sqrt
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
LLM_DIR = CLASSIFICATION_DIR / "llm_annotation"
CORRECTION_DIR = CLASSIFICATION_DIR / "human_correction"

GEMINI_DIR = LLM_DIR / "gemini"
PILOT_METRICS_PATH = GEMINI_DIR / "pilot/evaluation/critic_pilot_model_metrics.csv"
RUN_MANIFEST_PATH = GEMINI_DIR / "run_manifest.csv"
PRODUCTION_MANIFEST_PATH = GEMINI_DIR / "production/manifest.csv"
RANKED_REVIEW_PATH = CORRECTION_DIR / "frame_llm_ranked_review.xlsx"
AUDIT_PATH = CORRECTION_DIR / "frame_llm_residual_audit.xlsx"
FINAL_LABELS_PATH = CORRECTION_DIR / "frame_llm_correction_completed.csv"


## Pilot Calibration

The critic is calibrated only against the adjudicated human pilot and the matching high-reasoning Codex annotations. Gemini 3 Flash is accepted for production only if it produces valid hierarchical outputs, recovers at least 60% of known Codex errors among its top 40 ranked pilot cases, and shows no repeated systematic codebook blind spot.


In [2]:
if PILOT_METRICS_PATH.exists():
    pilot_metrics = pd.read_csv(PILOT_METRICS_PATH)
    display(pilot_metrics)
else:
    print("Pilot critic metrics are not available yet. Follow gemini_criticism_runbook.md.")


,requested_model,rows,known_errors,errors_found_top_40,recall_top_40,average_precision,passes_top_k_recall_gate
0,gemini-3-flash-preview,200,17,11,0.647059,0.337829,True


## Production Coverage and Usage

Critic chunks preserve direct provenance from validated Codex annotation batches while limiting each Gemini chunk to at most 10 rows. This check makes missing criticism, accidental batch changes, and future append-only extensions visible before human review.


In [3]:
if PRODUCTION_MANIFEST_PATH.exists():
    critic_manifest = pd.read_csv(PRODUCTION_MANIFEST_PATH)
    print(f"Critic input batches: {len(critic_manifest):,}")
    print(f"Critic input rows: {critic_manifest['rows'].sum():,}")
    display(critic_manifest.tail())
else:
    print("Production critic inputs have not been synchronized yet.")

if RUN_MANIFEST_PATH.exists():
    run_manifest = pd.read_csv(RUN_MANIFEST_PATH)
    production_runs = run_manifest.loc[run_manifest['dataset'].eq('production')].copy()
    print(f"Recorded production attempts: {len(production_runs):,}")
    display(production_runs.groupby(['requested_model', 'status'], dropna=False).size().rename('attempts').reset_index())
else:
    print("No Gemini critic run metadata is available yet.")


Critic input batches: 320
Critic input rows: 3,000


,critic_batch_name,source_batch_name,source_chunk_number,critic_batch_size,rows,annotation_id_start,annotation_id_end,source_input_sha256,source_output_sha256,critic_input_sha256,prompt_version,codebook_version
315,critic_batch_040_07.jsonl,annotator_batch_040.jsonl,7,10,10,llm_train_02961,llm_train_02970,1dcdc1247858b163ba3dc620d70cd3dd2d53c77bf4c874...,abdff9bc28edf571f652dd993097bc545fb989960c1194...,35854713fa7fdf2852b6c430f624792e55e11f4444aa78...,critic_v5,v0.4
316,critic_batch_040_08.jsonl,annotator_batch_040.jsonl,8,10,5,llm_train_02971,llm_train_02975,1dcdc1247858b163ba3dc620d70cd3dd2d53c77bf4c874...,abdff9bc28edf571f652dd993097bc545fb989960c1194...,d9bb81647af4e8176da5f73c13ab1598809177eb75621b...,critic_v5,v0.4
317,critic_batch_041_01.jsonl,annotator_batch_041.jsonl,1,10,10,llm_train_02976,llm_train_02985,a726d4d3dfb4c6911f179462cbd03add4717ba3852a8e0...,c336310b83a57cb59cccde922b83763480cf0da5ce3cc4...,4027ca84fe6b5e0e7ed4242d5911c55b16e8be4b61a120...,critic_v5,v0.4
318,critic_batch_041_02.jsonl,annotator_batch_041.jsonl,2,10,10,llm_train_02986,llm_train_02995,a726d4d3dfb4c6911f179462cbd03add4717ba3852a8e0...,c336310b83a57cb59cccde922b83763480cf0da5ce3cc4...,c3802235707bb5e1efadc07adc3fe46bc67746a11dc53c...,critic_v5,v0.4
319,critic_batch_041_03.jsonl,annotator_batch_041.jsonl,3,10,5,llm_train_02996,llm_train_03000,a726d4d3dfb4c6911f179462cbd03add4717ba3852a8e0...,c336310b83a57cb59cccde922b83763480cf0da5ce3cc4...,306e10a669e018bf7748019b79587fc6efaa746c267e82...,critic_v5,v0.4


Recorded production attempts: 325


,requested_model,status,attempts
0,gemini-3-flash-preview,invalid,5
1,gemini-3-flash-preview,valid,320


## Human Review Yield

The ranked workbook contains the critic's highest-priority cases. Review proceeds in score order and can stop before the maximum only after two consecutive completed 100-case waves each produce fewer than five corrections. The separate random audit is generated after the ranked stopping point and estimates errors among otherwise-unreviewed rows.


In [4]:
def parse_label(value: object) -> bool | None:
    if pd.isna(value) or str(value).strip().upper() == "NA":
        return None
    return str(value).strip().upper() == "TRUE"


def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (float("nan"), float("nan"))
    proportion = successes / total
    denominator = 1 + z**2 / total
    centre = (proportion + z**2 / (2 * total)) / denominator
    margin = z * sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
    return centre - margin, centre + margin


def review_summary(path: Path, source: str) -> pd.DataFrame:
    frame = pd.read_excel(path)
    reviewed = frame.loc[frame['review_status'].eq('reviewed')].copy()
    for axis in ['substantive_target_discourse', 'clinical_frame_present', 'lived_experience_frame_present']:
        reviewed[f'{axis}_changed'] = [
            parse_label(final) != parse_label(original)
            for final, original in zip(reviewed[f'final_{axis}'], reviewed[axis])
        ]
    reviewed['any_correction'] = reviewed[[f'{axis}_changed' for axis in ['substantive_target_discourse', 'clinical_frame_present', 'lived_experience_frame_present']]].any(axis=1)
    reviewed['review_source'] = source
    return reviewed

reviewed_parts = []
if RANKED_REVIEW_PATH.exists():
    ranked_reviewed = review_summary(RANKED_REVIEW_PATH, 'ranked')
    reviewed_parts.append(ranked_reviewed)
    ranked_reviewed['review_wave'] = ((ranked_reviewed['rank'] - 1) // 100) + 1
    display(ranked_reviewed.groupby('review_wave').agg(reviewed=('annotation_id', 'size'), corrections=('any_correction', 'sum'), mean_overall_error_prob=('overall_error_prob', 'mean')).reset_index())
else:
    print("Ranked review workbook is not available yet.")

if AUDIT_PATH.exists():
    audit_reviewed = review_summary(AUDIT_PATH, 'residual_audit')
    reviewed_parts.append(audit_reviewed)
    print(f"Residual audit reviewed: {len(audit_reviewed):,}")
    audit_corrections = int(audit_reviewed["any_correction"].sum())
    audit_low, audit_high = wilson_interval(audit_corrections, len(audit_reviewed))
    print(f"Residual audit corrections: {audit_corrections:,}")
    print(f"Residual audit error rate: {audit_corrections / len(audit_reviewed):.1%} (95% Wilson CI {audit_low:.1%}-{audit_high:.1%})")
else:
    print("Residual audit workbook is not available yet.")


,review_wave,reviewed,corrections,mean_overall_error_prob
0,1,100,86,0.9217
1,2,100,86,0.7403
2,3,100,62,0.1399
3,4,100,52,0.1000


Residual audit reviewed: 50
Residual audit corrections: 28
Residual audit error rate: 56.0% (95% Wilson CI 42.3%-68.8%)


## Final Corrected Training Labels

Finalization preserves Codex labels for every unreviewed row and applies only explicit human-reviewed changes. Codex confidence is retained because confidence is not used as a classifier feature or training weight.


In [5]:
if FINAL_LABELS_PATH.exists():
    final_labels = pd.read_csv(FINAL_LABELS_PATH)
    print(f"Final corrected rows: {len(final_labels):,}")
    print(f"Human-reviewed rows: {(final_labels['correction_source'] == 'human_review').sum():,}")
    display(final_labels['corrected_derived_frame'].value_counts(dropna=False).rename_axis('frame').reset_index(name='rows'))
else:
    print("Final corrected labels are not available yet. Complete both review workbooks and run finalize-corrections.")


Final corrected rows: 3,000
Human-reviewed rows: 450


,frame,rows
0,clinical_only,1227
1,non_substantive_or_insufficient,1008
2,lived_only,545
3,mixed,198
4,substantive_other,22
